# <span style="font-width:bold; font-size: 3rem; color:#1EB182;">**Garmin Companion**</span><span style="font-width:bold; font-size: 3rem; color:#333;"> - 02: Training Pipeline</span>

<span style="font-width:bold; font-size: 1.4rem;">Feature views, three models, and honest evaluation.</span>

> **Not medical advice.** This is a personal training-readiness & recovery monitoring system, not a diagnosis or injury-prediction tool.


We build three feature views, generate weak readiness labels and recovery-episode labels, train three models, and register them into **one** model directory for a single combined deployment (A8).

> ⚠️ **Read first — weak-label honesty (B1).** The readiness classifier is bootstrapped from a rule. A model trained on a rule using the *same inputs* just re-learns the rule, so its weak-label accuracy is **circular and meaningless**. We therefore (1) give the model *richer* features than the rule, and (2) evaluate against **manual feedback** using Cohen's κ, not weak-label F1. With ~6 months of one user's data, the **rule itself is the product**; learned models are what you graduate to with the multi-user entity model (B4).

In [ ]:
!pip install -U 'hopsworks[python,great_expectations]' scikit-learn --quiet

## <span style='color:#ff5f27'>📝 Imports &amp; connect</span>

In [ ]:
import warnings
import joblib
import numpy as np
import pandas as pd

import hopsworks
from features import weak_labels, recovery_episodes
import transformations as T

warnings.filterwarnings("ignore")

project = hopsworks.login()
fs = project.get_feature_store()
mr = project.get_model_registry()

daily_fg = fs.get_feature_group("fg_garmin_daily_summary_raw", 1)
sleep_fg = fs.get_feature_group("fg_garmin_sleep_raw", 1)
activity_fg = fs.get_feature_group("fg_garmin_activity_raw", 1)
baselines_fg = fs.get_feature_group("fg_recovery_baselines_daily", 1)
load_fg = fs.get_feature_group("fg_training_load_daily", 1)
stress_state_fg = fs.get_feature_group("fg_stress_state", 1)

## <span style='color:#ff5f27'>🏷️ 1. Daily features, weak readiness labels &amp; manual feedback</span>

All readiness features live at one **(user_id, date)** grain so the feature-view joins are clean 1:1 (no fan-out) and the online serving key is simply `(user_id, date)`. We fold the daily-summary + sleep columns the model needs into a single `fg_readiness_features_daily`.

In [ ]:
# Offline reads at the (user_id, date) grain.
daily = daily_fg.read().rename(columns={"summary_date": "date"})
sleepd = sleep_fg.read().rename(columns={"sleep_date": "date"})
base = baselines_fg.read()
load = load_fg.read()

features_daily = (
    daily[["user_id", "date", "resting_hr", "avg_stress",
           "body_battery_morning", "body_battery_recharge", "steps", "intensity_minutes"]]
    .merge(sleepd[["user_id", "date", "hrv_avg_sleep", "hrv_missing", "sleep_score",
                   "sleep_duration_min", "deep_sleep_min", "rem_sleep_min",
                   "sleep_debt_min", "deep_sleep_pct"]], on=["user_id", "date"], how="left")
)
features_daily["event_time"] = pd.to_datetime(features_daily["date"]) + pd.Timedelta(hours=8)

features_fg = fs.get_or_create_feature_group(
    name="fg_readiness_features_daily", version=1,
    description="Model-input daily features (user_id, date grain)",
    primary_key=["user_id", "date"], event_time="event_time", online_enabled=True,
)
features_fg.insert(features_daily)

In [ ]:
# Weak labels: assemble deltas + ACWR, then apply the rule (the V0 product).
frame = (
    features_daily.merge(base, on=["user_id", "date"], how="left")
    .merge(load[["user_id", "date", "ewma_acwr"]], on=["user_id", "date"], how="left")
)
labelled = weak_labels.add_weak_labels(frame)
labelled["event_time"] = pd.to_datetime(labelled["date"]) + pd.Timedelta(hours=8)

labels_fg = fs.get_or_create_feature_group(
    name="fg_readiness_labels", version=1, description="Weak readiness labels (V0 rule)",
    primary_key=["user_id", "date"], event_time="event_time", online_enabled=True,
)
labels_fg.insert(labelled[["user_id", "date", "event_time",
                           "readiness_class", "readiness_score", "hard_training_ok"]])

# Manual feedback (sparse, selection-biased) — for evaluation/calibration only (T5).
feedback_df = weak_labels.generate_demo_manual_feedback(labelled)
feedback_fg = fs.get_or_create_feature_group(
    name="fg_manual_feedback", version=1, description="Subjective manual feedback",
    primary_key=["user_id", "date"], event_time="event_time", online_enabled=True,
)
feedback_fg.insert(feedback_df)
print(f"{len(feedback_df)} feedback rows logged out of {len(labelled)} days (logged non-randomly).")

## <span style='color:#ff5f27'>🪟 2. Feature views (clean keyed joins &amp; model-dependent transforms)</span>

All readiness FGs share `(user_id, date)`, so we join `on=['user_id', 'date']` — one row per day, no fan-out. The view attaches the **delta** transforms (A1): pure feature-arithmetic over a feature and its stored baseline — skew-safe at train &amp; serve, no statistics needed.

In [ ]:
# --- fv_readiness_daily ----------------------------------------------------------
readiness_q = (
    labels_fg.select(["readiness_class"])
    .join(features_fg.select(["resting_hr", "avg_stress", "body_battery_morning",
                              "body_battery_recharge", "hrv_avg_sleep", "hrv_missing",
                              "sleep_score", "deep_sleep_min", "rem_sleep_min",
                              "sleep_debt_min", "deep_sleep_pct"]), on=["user_id", "date"])
    .join(baselines_fg.select(["rhr_28d_mean", "hrv_28d_mean", "stress_7d_mean"]),
          on=["user_id", "date"])
    .join(load_fg.select(["ewma_acute_load", "ewma_chronic_load", "ewma_acwr",
                          "days_since_last_hard_session", "training_strain_7d"]),
          on=["user_id", "date"])
)
# NOTE: logging_enabled is disabled — this cluster's Hive Metastore cannot create the
# prediction-logging feature group (COLUMNS_V2 insert fails server-side for any logging
# FG). The predictor's fv.log call is already wrapped in try/except so serving is
# unaffected; feature monitoring in notebook 4 runs on the FG/FV directly.
readiness_fv = fs.get_or_create_feature_view(
    name="fv_readiness_daily", version=1, query=readiness_q, labels=["readiness_class"],
    transformation_functions=[
        T.rhr_delta_28d("resting_hr", "rhr_28d_mean"),
        T.hrv_delta_28d_pct("hrv_avg_sleep", "hrv_28d_mean"),
    ],
)

In [ ]:
# --- fv_stress_anomaly_realtime (precomputed robust z-scores; unsupervised) -------
# logging_enabled disabled for the same cluster Metastore limitation noted above.
stress_q = stress_state_fg.select([
    "stress_zscore_vs_time_of_day", "hr_zscore_vs_time_of_day",
    "body_battery_slope_2h", "minutes_since_wake", "is_active",
])
stress_fv = fs.get_or_create_feature_view(
    name="fv_stress_anomaly_realtime", version=1, query=stress_q,
)

In [ ]:
# --- Recovery episodes (frozen pre-activity baseline, censored labels) ------------
acts = activity_fg.read()
morning = frame[["user_id", "date", "resting_hr", "hrv_avg_sleep", "sleep_score", "body_battery_morning"]]
episodes_df = recovery_episodes.build_recovery_episodes(acts, morning, base)
episodes_fg = fs.get_or_create_feature_group(
    name="fg_recovery_episodes", version=1, description="Post-workout recovery episodes + labels",
    primary_key=["user_id", "activity_id"], event_time="event_time", online_enabled=True,
)
episodes_fg.insert(episodes_df)

# --- fv_recovery_time_after_workout ----------------------------------------------
# Pre-workout features are already point-in-time-correct inside the episode FG
# (frozen pre-activity baseline, B2). Serving by (user_id, activity_id); no extra
# join needed (joining the daily-grain load FG on user_id alone would fan out).
# Numeric pre-workout features + the regression label only. We deliberately exclude
# `censored` and `recovered_within_24h_label` (they encode the outcome — leakage) and
# the string `activity_type` (would need encoding). recovery_hours_label is already
# NaN for censored episodes, so the target filter drops them.
recovery_q = episodes_fg.select(
    ["activity_load", "max_hr_pct", "zone4_5_minutes",
     "rhr_pre_baseline", "hrv_pre_baseline", "recovery_hours_label"]
)
recovery_fv = fs.get_or_create_feature_view(
    name="fv_recovery_time_after_workout", version=1, query=recovery_q,
    labels=["recovery_hours_label"], logging_enabled=True,
)

> 🧷 **Spine groups (A3).** Our recovery labels already live in a feature group, so the view above serves online directly. When labels arrive from *outside* the feature store, the idiomatic point-in-time pattern is a **spine group**:
> ```python
> spine = fs.get_or_create_spine_group(
>     name='recovery_spine', version=1,
>     primary_key=['user_id', 'activity_id'], event_time='event_time',
>     dataframe=labels_df,  # external labels + event_time
> )
> query = spine.select(['recovery_hours_label']).join(load_fg.select([...]), on=['user_id'])
> # PIT join picks each feature row as-of the spine's event_time — no fan-out.
> ```

## <span style='color:#ff5f27'>🤖 3. Train the three models</span>

We hold out 20%. ⚠️ For a **strict forward-chaining** split (adjacent days leak via autocorrelation, T6) pass `train_end` / `test_start` dates to `train_test_split` rather than `test_size`. We use plain scikit-learn (XGBoost/LightGBM are a later upgrade, B4).

In [ ]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest, GradientBoostingRegressor

# --- Readiness classifier (richer features than the rule) -------------------------
X_train, X_test, y_train, y_test = readiness_fv.train_test_split(test_size=0.2)
readiness_model = RandomForestClassifier(n_estimators=200, random_state=42)
readiness_model.fit(X_train, y_train.values.ravel())
print("readiness holdout accuracy (vs WEAK labels — circular, see caveat):",
      round(readiness_model.score(X_test, y_test.values.ravel()), 3))

In [ ]:
# Honest evaluation: rule vs MANUAL feedback via Cohen's kappa (B1).
fb = feedback_fg.read().merge(labelled[["user_id", "date", "readiness_class"]],
                              on=["user_id", "date"], how="inner")
rule_cls = fb["readiness_class"].tolist()
felt_cls = [weak_labels.perceived_to_class(p) for p in fb["perceived_recovery_1_5"]]
print("Cohen's kappa (rule vs perceived recovery):",
      round(weak_labels.cohens_kappa(rule_cls, felt_cls), 3),
      f"on {len(fb)} self-rated days (logged non-randomly).")

In [ ]:
# --- Stress anomaly detector (unsupervised IsolationForest) -----------------------
# A label-free feature view still returns the 4-tuple (X_train, X_test, y_train,
# y_test) in this SDK; the label halves are simply empty, so we discard them.
Xs, _, _, _ = stress_fv.train_test_split(test_size=0.2)
stress_model = IsolationForest(contamination=0.05, random_state=42)
stress_model.fit(Xs.fillna(0))
print("stress model trained on", len(Xs), "epochs")

In [ ]:
# --- Recovery-time regressor (drop censored / unknown labels for the target) ------
Xr, _, yr, _ = recovery_fv.train_test_split(test_size=0.2)
mask = yr.iloc[:, 0].notna()
recovery_model = GradientBoostingRegressor(random_state=42)
recovery_model.fit(Xr[mask].fillna(0), yr[mask].iloc[:, 0])
print(f"recovery model trained on {int(mask.sum())} uncensored episodes "
      f"({len(yr) - int(mask.sum())} censored/unknown excluded from the target).")

> 📈 *Optional:* a descriptive **Kaplan–Meier** curve over the recovery episodes honestly visualizes censoring without fitting a fragile survival model at this sample size (panel decision).

## <span style='color:#ff5f27'>📦 4. Register one combined model</span>

All three pickles go into **one** model directory so a single deployment can load them (A8). The combined model is linked to the readiness feature view via provenance (`feature_view=`), which the predictor resolves with `model.get_feature_view()`.

In [ ]:
import os
from hsml.schema import Schema
from hsml.model_schema import ModelSchema

model_dir = "garmin_models"
os.makedirs(model_dir, exist_ok=True)
joblib.dump(readiness_model, f"{model_dir}/readiness_model.pkl")
joblib.dump(stress_model, f"{model_dir}/stress_model.pkl")
joblib.dump(recovery_model, f"{model_dir}/recovery_model.pkl")

model_schema = ModelSchema(input_schema=Schema(X_train), output_schema=Schema(y_train))
combined = mr.python.create_model(
    name="garmin_combined", version=1,
    metrics={"readiness_holdout_acc": float(readiness_model.score(X_test, y_test.values.ravel()))},
    model_schema=model_schema,
    feature_view=readiness_fv,
    input_example=[{"user_id": "javier", "date": "2024-06-30"}],
    description="Combined Garmin readiness / stress-anomaly / recovery model",
)
combined.save(model_dir)

✅ Models registered. Deploy and serve in **`3_garmin_inference_pipeline.ipynb`**.